# Multimodal third encoder: FLAVA on MSCOCO and SNLI-VE (N=1000)

Completes the multimodal block to three encoders spanning three pretraining
paradigms: contrastive (CLIP), sigmoid-contrastive (SigLIP), and multitask
multimodal (FLAVA, `facebook/flava-full`). Joint embeddings are the
concatenation of the image-tower CLS and text-tower CLS states, each L2
normalized, 768 + 768 = 1536, mirroring the SigLIP treatment.

Dataset construction follows the existing multimodal configurations exactly:
SNLI-VE balanced 400/class train and 90/class test; MSCOCO 600 train and 130
test images expanded to matched/mismatched pairs with `RandomState(42)`; both
subsampled to 1000/200 with seed 42. Primary and fallback Hub sources are the
two used by your CLIP and SigLIP runs.

Each dataset checkpoints after embeddings and after the kernel (~85 min each,
about 3.5 h total); re-running skips completed stages.

In [ ]:
!pip install pennylane pennylane-lightning transformers datasets --quiet

import pennylane as qml
import numpy as np
import torch, time, json, pickle, os, shutil, warnings
from PIL import Image
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics.pairwise import rbf_kernel
warnings.filterwarnings('ignore')
print(f"PennyLane {qml.__version__} | torch {torch.__version__}")
DEVICE_Q = qml.device('lightning.qubit', wires=8)
SEED = 42

---
## Step 0: Drive storage

In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/qml_project'
else:
    SAVE_DIR = 'qml_project'
os.makedirs(SAVE_DIR, exist_ok=True)

def checkpoint(filename, save_fn):
    save_fn(filename)
    dst = os.path.join(SAVE_DIR, os.path.basename(filename))
    if os.path.abspath(filename) != os.path.abspath(dst):
        shutil.copy(filename, dst)
    print(f"  saved -> {dst}")

def find_file(fname):
    dp = os.path.join(SAVE_DIR, fname)
    return dp if os.path.exists(dp) else (fname if os.path.exists(fname) else None)
print("ready")

---
## Step 1: Dataset loaders (construction identical to CLIP/SigLIP runs)

In [ ]:
def load_snlive():
    # balanced 400/class train, 90/class test; sedrickkeh primary (SigLIP source),
    # HuggingFaceM4 fallback (CLIP source); string labels mapped
    from datasets import load_dataset
    STR2INT = {'entailment': 0, 'neutral': 1, 'contradiction': 2}
    last_err = None
    for src in ["sedrickkeh/snli-ve", "HuggingFaceM4/SNLI-VE"]:
        try:
            print(f"loading {src}...")
            ds = load_dataset(src)
            break
        except Exception as e:
            print(f"  {src} failed: {type(e).__name__}"); last_err = e
    else:
        raise last_err
    cols = ds['train'].column_names
    img_col = 'image' if 'image' in cols else [c for c in cols if 'image' in c.lower() or 'img' in c.lower() or c=='premise'][0]
    txt_col = 'hypothesis' if 'hypothesis' in cols else [c for c in cols if 'hypo' in c.lower() or 'sentence' in c.lower() or 'caption' in c.lower()][0]
    lbl_col = 'label' if 'label' in cols else [c for c in cols if 'label' in c.lower() or 'gold' in c.lower()][0]
    def get_label(s):
        l = s[lbl_col]
        return STR2INT.get(l, l) if isinstance(l, str) else l
    def load_balanced(split, n_per_class):
        imgs, txts, lbls = [], [], []
        counts = {0: 0, 1: 0, 2: 0}
        for s in ds[split]:
            l = get_label(s)
            if l not in (0, 1, 2) or counts[l] >= n_per_class: continue
            img = s[img_col]
            if not isinstance(img, Image.Image): continue
            imgs.append(img.convert('RGB')); txts.append(s[txt_col]); lbls.append(l)
            counts[l] += 1
            if all(v >= n_per_class for v in counts.values()): break
        return imgs, txts, np.array(lbls)
    tr_i, tr_t, tr_l = load_balanced('train', 400)
    test_split = 'test' if 'test' in ds else 'validation'
    te_i, te_t, te_l = load_balanced(test_split, 90)
    assert len(tr_i) == 1200 and len(te_i) == 270, f"got {len(tr_i)}/{len(te_i)}"
    return ('pairs_direct', (tr_i, tr_t, tr_l), (te_i, te_t, te_l))

def load_mscoco():
    # 600 train + 130 test images with one caption each; matched/mismatched
    # pairs via RandomState(42); nlphuji primary (CLIP source), lmms-lab fallback
    from datasets import load_dataset
    all_imgs, all_caps = [], []
    try:
        print("loading nlphuji/mscoco_2014_5k_test_image_text_retrieval...")
        ds = load_dataset("nlphuji/mscoco_2014_5k_test_image_text_retrieval", split='test')
        cols = ds.column_names
        img_col = 'image' if 'image' in cols else [c for c in cols if 'image' in c.lower()][0]
        cap_col = 'caption' if 'caption' in cols else ('captions' if 'captions' in cols else [c for c in cols if 'caption' in c.lower()][0])
        for s in ds:
            cap = s[cap_col]
            if isinstance(cap, list): cap = cap[0] if len(cap) else ""
            img = s[img_col]
            if not isinstance(img, Image.Image): continue
            all_imgs.append(img.convert('RGB')); all_caps.append(str(cap))
            if len(all_imgs) >= 780: break
    except Exception as e:
        print(f"  nlphuji failed ({type(e).__name__}); lmms-lab fallback")
        ds_stream = load_dataset("lmms-lab/COCO-Caption", split="val", streaming=True)
        for s in ds_stream:
            img = s.get('image'); cap = s.get('answer')
            if isinstance(cap, list) and len(cap): cap = cap[0]
            if not isinstance(img, Image.Image): continue
            all_imgs.append(img.convert('RGB')); all_caps.append(str(cap))
            if len(all_imgs) >= 780: break
    assert len(all_imgs) >= 730, f"only {len(all_imgs)} images collected"
    rng = np.random.RandomState(42)
    def build_pairs(idxs):
        pairs = []
        for i in idxs:
            pairs.append((i, i, 1))
            j = rng.choice([k for k in idxs if k != i])
            pairs.append((i, int(j), 0))
        rng.shuffle(pairs)
        return pairs
    train_pairs = build_pairs(list(range(600)))
    test_pairs  = build_pairs(list(range(600, 730)))
    assert len(train_pairs) == 1200 and len(test_pairs) == 260
    return ('pairs_indexed', (all_imgs, all_caps), (train_pairs, test_pairs))

CONFIGS = {
    'snlive_flava': {'loader': load_snlive, 'metric': 'macro',  'dataset': 'snlive'},
    'mscoco_flava': {'loader': load_mscoco, 'metric': 'binary', 'dataset': 'mscoco'},
}
print("loaders defined:", list(CONFIGS))

---
## Step 2: FLAVA joint encoder (image CLS + text CLS, each L2 normalized, 1536)

In [ ]:
from transformers import FlavaProcessor, FlavaModel
flava_proc  = FlavaProcessor.from_pretrained('facebook/flava-full')
flava_model = FlavaModel.from_pretrained('facebook/flava-full').eval()
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
flava_model = flava_model.to(dev)
print(f"FLAVA on {dev}")

@torch.no_grad()
def encode_images(images, batch=32):
    out = []
    for i in range(0, len(images), batch):
        inp = flava_proc(images=images[i:i+batch], return_tensors='pt').to(dev)
        h = flava_model.get_image_features(pixel_values=inp['pixel_values']).last_hidden_state[:, 0, :]
        h = h / torch.linalg.norm(h, dim=-1, keepdim=True)
        out.append(h.cpu().numpy())
    return np.vstack(out)

@torch.no_grad()
def encode_texts(texts, batch=32):
    out = []
    for i in range(0, len(texts), batch):
        inp = flava_proc(text=list(texts[i:i+batch]), return_tensors='pt',
                         padding=True, truncation=True, max_length=77).to(dev)
        h = flava_model.get_text_features(input_ids=inp['input_ids'],
                                          attention_mask=inp['attention_mask']).last_hidden_state[:, 0, :]
        h = h / torch.linalg.norm(h, dim=-1, keepdim=True)
        out.append(h.cpu().numpy())
    return np.vstack(out)

# smoke test: one image, one text
_img = Image.new('RGB', (224, 224), (128, 128, 128))
_e_i = encode_images([_img]); _e_t = encode_texts(["a smoke test"])
assert _e_i.shape == (1, 768) and _e_t.shape == (1, 768), f"{_e_i.shape} {_e_t.shape}"
assert np.isclose(np.linalg.norm(_e_i[0]), 1.0, atol=1e-4)
print("FLAVA towers verified: 768 + 768 -> 1536 joint")

---
## Step 3: Quantum kernel and diagnostics (identical definitions to the grid)

In [ ]:
def iqp_feature_map(x, wires):
    n = len(wires)
    for w in wires: qml.Hadamard(wires=w)
    for i, w in enumerate(wires): qml.RZ(x[i], wires=w)
    for i in range(n - 1):
        qml.CNOT(wires=[wires[i], wires[i+1]])
        qml.RZ(x[i] * x[i+1], wires=wires[i+1])
        qml.CNOT(wires=[wires[i], wires[i+1]])

@qml.qnode(DEVICE_Q)
def kernel_circuit(x1, x2):
    iqp_feature_map(x1, wires=range(8))
    qml.adjoint(iqp_feature_map)(x2, wires=range(8))
    return qml.probs(wires=range(8))

quantum_kernel = lambda a, b: float(kernel_circuit(a, b)[0])
assert np.isclose(quantum_kernel(np.linspace(0, 3, 8), np.linspace(0, 3, 8)), 1.0, atol=1e-5)

def kta(K, y):
    y = np.asarray(y); n = len(y)
    Y = np.zeros((n, n))
    for c in np.unique(y):
        m = (y == c).astype(float); Y += np.outer(m, m)
    H = np.eye(n) - np.ones((n, n))/n
    Kc, Yc = H @ K @ H, H @ Y @ H
    return float(np.sum(Kc*Yc) / np.sqrt(np.sum(Kc*Kc) * np.sum(Yc*Yc)))

def concentration(K):
    mu = np.linalg.eigvalsh(K); mu = np.maximum(mu, 0)
    lam = mu / mu.sum()
    return float(np.exp(-np.sum(lam * np.log(lam + 1e-15))) / K.shape[0])

def geometric_difference_exact(K_Q, K_C, lam=1e-6):
    n = K_Q.shape[0]
    K_Q = K_Q * n / np.trace(K_Q); K_C = K_C * n / np.trace(K_C)
    w, V = np.linalg.eigh(K_Q); w = np.maximum(w, 0)
    sq = (V * np.sqrt(w)) @ V.T
    return float(np.sqrt(max(np.linalg.eigvalsh(sq @ np.linalg.inv(K_C + lam*n*np.eye(n)) @ sq).max(), 0.0)))

def geometric_difference_proxy(K_Q, K_C):
    eff = lambda K: (np.trace(K)**2) / (np.sum(K**2) + 1e-10)
    return float(eff(K_Q) / (eff(K_C) + 1e-10))
print("pipeline pieces verified")

---
## Step 4: Run both configurations end to end

In [ ]:
for TAG, cfg in CONFIGS.items():
    print(f"\n{'='*60}\n{TAG}\n{'='*60}")
    if find_file(f'{TAG}_grid_record.json'):
        print("record exists on Drive; skipping"); continue
    AVG = cfg['metric']
    def fscore(yt, yp): return float(f1_score(yt, yp, average=AVG))

    # ---- embeddings (checkpointed) ----
    emb_f = f'{TAG}_embeddings.npz'
    hit = find_file(emb_f)
    if hit:
        z = np.load(hit)
        X_tr, y_tr, X_te, y_te = z['X_tr_subset'], z['y_tr_subset'], z['X_te_subset'], z['y_te_subset']
        print("embeddings loaded from checkpoint")
    else:
        kind, train_data, test_data = cfg['loader']()
        t0 = time.time()
        if kind == 'pairs_direct':      # snlive: per-sample image + hypothesis
            tr_i, tr_t, tr_l = train_data; te_i, te_t, te_l = test_data
            train_emb = np.concatenate([encode_images(tr_i), encode_texts(tr_t)], axis=1)
            test_emb  = np.concatenate([encode_images(te_i), encode_texts(te_t)], axis=1)
            train_labels, test_labels = tr_l, te_l
        else:                            # mscoco: cached towers, assemble pairs
            all_imgs, all_caps = train_data
            train_pairs, test_pairs = test_data
            img_cache = encode_images(all_imgs)
            txt_cache = encode_texts(all_caps)
            def assemble(pairs):
                X, y = [], []
                for ii, ci, lb in pairs:
                    X.append(np.concatenate([img_cache[ii], txt_cache[ci]])); y.append(lb)
                return np.array(X), np.array(y)
            train_emb, train_labels = assemble(train_pairs)
            test_emb,  test_labels  = assemble(test_pairs)
        print(f"encoded in {(time.time()-t0)/60:.1f} min | train {train_emb.shape} test {test_emb.shape}")
        assert train_emb.shape[1] == 1536

        pca = PCA(n_components=8, random_state=SEED).fit(train_emb)
        P_tr, P_te = pca.transform(train_emb), pca.transform(test_emb)
        print(f"PCA-8 variance: {pca.explained_variance_ratio_.sum():.4f}")
        scaler = MinMaxScaler(feature_range=(0, np.pi)).fit(P_tr)
        Xtr_all = np.clip(scaler.transform(P_tr), 0, np.pi)
        Xte_all = np.clip(scaler.transform(P_te), 0, np.pi)

        idx_tr = np.random.RandomState(SEED).choice(len(Xtr_all), 1000, replace=False)
        idx_te = np.random.RandomState(SEED).choice(len(Xte_all), 200, replace=False)
        X_tr, y_tr = Xtr_all[idx_tr], train_labels[idx_tr]
        X_te, y_te = Xte_all[idx_te], test_labels[idx_te]
        checkpoint(emb_f, lambda p: np.savez(p, X_train=Xtr_all, X_test=Xte_all,
                                             y_train=train_labels, y_test=test_labels,
                                             X_tr_subset=X_tr, y_tr_subset=y_tr,
                                             X_te_subset=X_te, y_te_subset=y_te))
    assert X_tr.shape == (1000, 8) and X_te.shape == (200, 8)
    assert X_tr.min() >= -1e-9 and X_tr.max() <= np.pi + 1e-9
    print(f"class balance train: {dict(zip(*np.unique(y_tr, return_counts=True)))}")

    # ---- classical baselines ----
    baselines, fitted = {}, {}
    for name, clf in [('Linear', SVC(kernel='linear', C=1.0, random_state=SEED)),
                      ('RBF', SVC(kernel='rbf', C=1.0, gamma='scale', random_state=SEED)),
                      ('Polynomial', SVC(kernel='poly', degree=3, coef0=1, C=1.0, gamma='scale', random_state=SEED))]:
        yp = clf.fit(X_tr, y_tr).predict(X_te)
        fitted[name] = clf
        baselines[name] = {'accuracy': float(accuracy_score(y_te, yp)), 'f1': fscore(y_te, yp)}
        print(f"  {name:<12} f1={baselines[name]['f1']:.4f}")
    best_nm = max(baselines, key=lambda k: baselines[k]['f1'])
    best_f1 = baselines[best_nm]['f1']

    # ---- quantum kernel (checkpointed) ----
    ker_f = f'{TAG}_kernel_matrices.npz'
    hit = find_file(ker_f)
    if hit:
        z = np.load(hit); K_train, K_test = z['K_train_quantum'], z['K_test_quantum']
        elapsed = float(z['elapsed_min']) if 'elapsed_min' in z.files else -1
        print("kernel loaded from checkpoint")
    else:
        print("computing kernel (~85 min)...")
        t0 = time.time()
        K_train = qml.kernels.kernel_matrix(X_tr, X_tr, quantum_kernel)
        K_test  = qml.kernels.kernel_matrix(X_te, X_tr, quantum_kernel)
        elapsed = (time.time()-t0)/60
        print(f"done in {elapsed:.1f} min")
        checkpoint(ker_f, lambda p: np.savez(p, K_train_quantum=K_train, K_test_quantum=K_test,
                                             X_tr=X_tr, y_tr=y_tr, X_te=X_te, y_te=y_te,
                                             elapsed_min=elapsed))

    # ---- QSVM, diagnostics, CI, learning curve ----
    qsvm = SVC(kernel='precomputed', C=1.0).fit(K_train, y_tr)
    y_pred = qsvm.predict(K_test)
    f1_q = fscore(y_te, y_pred); acc_q = float(accuracy_score(y_te, y_pred))
    gap = f1_q - best_f1
    print(f"quantum F1={f1_q:.4f} | best classical {best_f1:.4f} ({best_nm}) | gap {gap:+.4f}")

    Krbf = rbf_kernel(X_tr, X_tr, gamma=1.0/X_tr.shape[1])
    A_Q, A_C = kta(K_train, y_tr), kta(Krbf, y_tr)
    rho_ = concentration(K_train)
    g_ex = geometric_difference_exact(K_train, Krbf)
    g_px = geometric_difference_proxy(K_train, Krbf)
    lam_sweep = {f"{l:.0e}": round(geometric_difference_exact(K_train, Krbf, l), 2)
                 for l in [1e-4, 1e-5, 1e-6, 1e-7]}
    off_q = K_train[~np.eye(1000, dtype=bool)].mean()
    off_c = Krbf[~np.eye(1000, dtype=bool)].mean()
    print(f"A_Q={A_Q:.4f} A_C={A_C:.4f} rho={rho_:.4f} g={g_ex:.2f} (proxy {g_px:.2f})")
    print(f"lambda sweep: {lam_sweep}")

    rng = np.random.RandomState(SEED)
    yc_pred = fitted[best_nm].predict(X_te)
    boot = []
    for _ in range(1000):
        idx = rng.randint(0, len(y_te), len(y_te))
        if len(np.unique(y_te[idx])) < 2: continue
        boot.append(fscore(y_te[idx], y_pred[idx]) - fscore(y_te[idx], yc_pred[idx]))
    ci_lo, ci_hi = np.percentile(boot, [2.5, 97.5])
    print(f"gap CI [{ci_lo:+.4f}, {ci_hi:+.4f}]")

    sizes = [50, 100, 200, 400, 700, 1000]
    lc = {'Quantum (IQP)': [], 'Linear': [], 'RBF': []}
    for nn in sizes:
        yp = SVC(kernel='precomputed', C=1.0).fit(K_train[:nn,:nn], y_tr[:nn]).predict(K_test[:,:nn])
        lc['Quantum (IQP)'].append(fscore(y_te, yp))
        lc['Linear'].append(fscore(y_te, SVC(kernel='linear', C=1.0, random_state=SEED).fit(X_tr[:nn], y_tr[:nn]).predict(X_te)))
        lc['RBF'].append(fscore(y_te, SVC(kernel='rbf', C=1.0, gamma='scale', random_state=SEED).fit(X_tr[:nn], y_tr[:nn]).predict(X_te)))

    record = {'tag': TAG, 'domain': 'multimodal', 'encoder': 'flava', 'dataset': cfg['dataset'],
              'metric': AVG, 'embedding_type': 'flava_joint_1536',
              'n_qubits': 8, 'n_train': 1000, 'n_test': 200,
              'compute_time_min': round(elapsed, 1),
              'quantum_accuracy': acc_q, 'quantum_f1': f1_q,
              'best_classical_f1': best_f1, 'best_classical_kernel': best_nm,
              'gap': gap, 'kta': A_Q, 'c_kta': A_C, 'concentration': rho_,
              'geometric_difference_exact': g_ex, 'geometric_difference_proxy': g_px,
              'exact_g_lambda_sweep': lam_sweep,
              'offdiag_quantum': float(off_q), 'offdiag_rbf': float(off_c),
              'ci_lo': float(ci_lo), 'ci_hi': float(ci_hi),
              'excludes_zero': bool(ci_lo > 0 or ci_hi < 0),
              'classical_baselines': baselines,
              'learning_curve_sizes': sizes, 'learning_curve_results': lc}
    checkpoint(f'{TAG}_grid_record.json', lambda p: json.dump(record, open(p, 'w'), indent=2))

print("\nAll done. Upload snlive_flava_grid_record.json and mscoco_flava_grid_record.json.")